# This file will find the closest school for each properties as the extra features, store school location & distance

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from sklearn.neighbors import BallTree
from geopy.distance import great_circle

In [2]:
# read the files record propertry and school respectively
property_data_origin = pd.read_csv("../data/curated/merged_data/merged_data.csv")
school_data = pd.read_csv("../data/test/school_location.csv") # change path later
# only select the needed features for running faster
school_data =  school_data[['School_Name','geometry']]
property_data =  property_data_origin[['name','coordinates']]
property_data

,name,coordinates
0,31 Chittagong Drive Clyde North VIC 3978,"[-38.1053122, 145.3570863]"
1,50 Elmtree Crescent Clyde North VIC 3978,"[-38.0825712, 145.3561984]"
2,7 Mortdale Lane Clyde North VIC 3978,"[-38.0961758, 145.3800644]"
3,54 Walhallow Drive Clyde North VIC 3978,"[-38.1133324, 145.3457396]"
4,10 Sicily Road Clyde North VIC 3978,"[-38.1295789, 145.3642993]"
...,...,...
8797,61 Tongue Street Yarraville VIC 3013,"[-37.8131463, 144.8909053]"
8798,47 Mill Avenue Yarraville VIC 3013,"[-37.8222822, 144.872198]"
8799,12 Adeney Street Yarraville VIC 3013,"[-37.8163817, 144.8666543]"
8800,229B Somerville Road Yarraville VIC 3013,"[-37.8124289, 144.8779569]"


In [3]:
school_data

,School_Name,geometry
0,Alberton Primary School,"[-38.61771, 146.6666]"
1,Allansford and District Primary School,"[-38.38628, 142.59039]"
2,Avoca Primary School,"[-37.0845, 143.47565]"
3,Avenel Primary School,"[-36.90137, 145.23472]"
4,Warrandyte Primary School,"[-37.74268, 145.21398]"
...,...,...
2297,Plenty River College,"[-37.64875, 145.08148]"
2298,Holy Cross Catholic Primary School,"[-37.53046, 144.9052]"
2299,Sidrah Gardens School,"[-37.97324, 145.31589]"
2300,Mountain District Community College,"[-37.88319, 145.29327]"


In [4]:
# This function will accept a string to parse coordinate string into point object
def parse_coordinates(coord_str):
    parts = coord_str.strip('[]').split(',')
    return (float(parts[0]), float(parts[1]))

# parse coordinate string into point object for both property & school data
closest_school = []
school_distance = []
property_coordinates = property_data['coordinates'].apply(parse_coordinates)
all_school_coordinates = school_data['geometry'].apply(parse_coordinates)

# find the closest school for each property
for property_coord in property_coordinates:
    # By default: no nearest school and the distance is positive infinity.
    min_distance = float('inf')
    closest_school_geo = None
    
    for i, school_coord in enumerate(all_school_coordinates):
        # check if the value of the coordinate point is valid
        if np.isnan(property_coord).any() or np.isnan(school_coord).any():
            continue
        # claculate the distance between property and school, update if it is smallest
        distance = great_circle(property_coord, school_coord).kilometers
        if distance < min_distance:
            min_distance = distance
            closest_school_geo = school_data.loc[i, 'geometry']
    
    # add feature to store closest school for the property
    closest_school.append(closest_school_geo)
    school_distance.append(min_distance)
property_data['closest_school'] = closest_school
property_data['school_distance(KM)'] = school_distance
property_data

/tmp/ipykernel_120841/3654868371.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  property_data['closest_school'] = closest_school
/tmp/ipykernel_120841/3654868371.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  property_data['school_distance(KM)'] = school_distance


,name,coordinates,closest_school,school_distance(KM)
0,31 Chittagong Drive Clyde North VIC 3978,"[-38.1053122, 145.3570863]","[-38.10602, 145.37876]",1.898006
1,50 Elmtree Crescent Clyde North VIC 3978,"[-38.0825712, 145.3561984]","[-38.08468, 145.3638]",0.705427
2,7 Mortdale Lane Clyde North VIC 3978,"[-38.0961758, 145.3800644]","[-38.10602, 145.37876]",1.100561
3,54 Walhallow Drive Clyde North VIC 3978,"[-38.1133324, 145.3457396]","[-38.11488, 145.33828]",0.674921
4,10 Sicily Road Clyde North VIC 3978,"[-38.1295789, 145.3642993]","[-38.12955, 145.33886]",2.225124
...,...,...,...,...
8797,61 Tongue Street Yarraville VIC 3013,"[-37.8131463, 144.8909053]","[-37.8137, 144.8899]",0.107655
8798,47 Mill Avenue Yarraville VIC 3013,"[-37.8222822, 144.872198]","[-37.82104, 144.87443]",0.239821
8799,12 Adeney Street Yarraville VIC 3013,"[-37.8163817, 144.8666543]","[-37.8126, 144.87466]",0.819385
8800,229B Somerville Road Yarraville VIC 3013,"[-37.8124289, 144.8779569]","[-37.8126, 144.87466]",0.290245


In [5]:
# store closest school information to the merged data
property_data_origin['closest_school'] = closest_school
property_data_origin['school_distance(KM)'] = school_distance
# save as a CSV file
property_data_origin.to_csv("../data/curated/merged_data/merged_data_with_facility.csv", index=False)
merged_data_with_facility = pd.read_csv("../data/curated/merged_data/merged_data_with_facility.csv")
merged_data_with_facility

,name,rental_price,num_bedroom,num_bathroom,num_parking,postcode,coordinates,property_geometry,sa2_code,sa2_name,sa2_geometry,personal_income,avg_income_growth_rate(%),2021_population,avg_pop_growth_rates(%),crime_rate(%),closest_school,school_distance(KM)
0,31 Chittagong Drive Clyde North VIC 3978,575.0,4,2,2.0,3978.0,"[-38.1053122, 145.3570863]",POINT (145.3570863 -38.1053122),212031556.0,Clyde North - South,POLYGON ((145.37034567475234 -38.0937851973191...,68495.004299,3.028540,15038.0,129.962525,0.122595,"[-38.10602, 145.37876]",1.898006
1,50 Elmtree Crescent Clyde North VIC 3978,560.0,4,2,2.0,3978.0,"[-38.0825712, 145.3561984]",POINT (145.3561984 -38.0825712),212031555.0,Clyde North - North,POLYGON ((145.3346630127421 -38.07820964472685...,68495.004299,3.028540,10652.0,29.771333,0.122595,"[-38.08468, 145.3638]",0.705427
2,7 Mortdale Lane Clyde North VIC 3978,490.0,2,2,1.0,3978.0,"[-38.0961758, 145.3800644]",POINT (145.3800644 -38.0961758),212031556.0,Clyde North - South,POLYGON ((145.37034567475234 -38.0937851973191...,68495.004299,3.028540,15038.0,129.962525,0.122595,"[-38.10602, 145.37876]",1.100561
3,54 Walhallow Drive Clyde North VIC 3978,540.0,4,2,1.0,3978.0,"[-38.1133324, 145.3457396]",POINT (145.3457396 -38.1133324),212031556.0,Clyde North - South,POLYGON ((145.37034567475234 -38.0937851973191...,68495.004299,3.028540,15038.0,129.962525,0.122595,"[-38.11488, 145.33828]",0.674921
4,10 Sicily Road Clyde North VIC 3978,520.0,4,2,2.0,3978.0,"[-38.1295789, 145.3642993]",POINT (145.3642993 -38.1295789),212031303.0,Cranbourne South,POLYGON ((145.3254642788471 -38.12742071438774...,67445.099591,2.233051,17641.0,14.250357,0.197962,"[-38.12955, 145.33886]",2.225124
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8797,61 Tongue Street Yarraville VIC 3013,630.0,2,1,0.0,3013.0,"[-37.8131463, 144.8909053]",POINT (144.8909053 -37.8131463),213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,92944.746058,3.965740,15651.0,0.003162,0.197223,"[-37.8137, 144.8899]",0.107655
8798,47 Mill Avenue Yarraville VIC 3013,730.0,4,3,2.0,3013.0,"[-37.8222822, 144.872198]",POINT (144.872198 -37.8222822),213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,92944.746058,3.965740,15651.0,0.003162,0.197223,"[-37.82104, 144.87443]",0.239821
8799,12 Adeney Street Yarraville VIC 3013,450.0,3,1,2.0,3013.0,"[-37.8163817, 144.8666543]",POINT (144.8666543 -37.8163817),213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,92944.746058,3.965740,15651.0,0.003162,0.197223,"[-37.8126, 144.87466]",0.819385
8800,229B Somerville Road Yarraville VIC 3013,300.0,1,1,0.0,3013.0,"[-37.8124289, 144.8779569]",POINT (144.8779569 -37.8124289),213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,92944.746058,3.965740,15651.0,0.003162,0.197223,"[-37.8126, 144.87466]",0.290245
